In [0]:
%sql
CREATE CONNECTION IF NOT EXISTS earthquake_test
TYPE HTTP
OPTIONS
(
    host = "https://earthquake.usgs.gov",
    port = 443,
    base_path = "/earthquakes/feed/v1.0/",
    bearer_token = 'NA'
)

In [0]:
%py
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
conn = w.connections.get("earthquake_test")
print(conn)
base_url = f"{conn.options['host']}{conn.options['base_path']}"



In [0]:
display(spark.sql('SHOW CATALOGS'))

In [0]:
dbutils.widgets.text('catalog_name', 'earthquake', 'earthquake')
catalog_name = dbutils.widgets.get('catalog_name')
print(catalog_name)

In [0]:
spark.sql(
    f"USE CATALOG {catalog_name}"
)
spark.sql(
    "USE SCHEMA bronze"
)
spark.sql(
    "CREATE VOLUME IF NOT EXISTS earthquake_data"
)

In [0]:
import requests
import json
import datetime

url = f"{base_url}/summary/all_day.geojson"
response = requests.get(url)

if response.status_code != 200:
    raise Exception(f"Error in getting data from URL: {url}")

data = response.json()
current_date = datetime.datetime.now().strftime("%Y-%m-%d")
dbutils.fs.put(
    f"/Volumes/{catalog_name}/bronze/earthquake_data/earthquake_data_{current_date}.json",
    json.dumps(data),
    overwrite=True,
)